# ELP capture (preview + photos)

**The preview runs on a background thread.** A `while` loop in the cell would block the
kernel, and a blocked kernel never runs a button callback, so the button would be dead until
the loop finished. The thread keeps the kernel free; the buttons work while frames arrive.

**Ask for a mode that exists.** `elp.open_elp(strict=True)` raises when the driver grants
something other than what was asked. Measured modes are in `elp_camera.json`:

| mode | fps | note |
|---|---|---|
| 1280×800 | 121 | native |
| 1280×720 | 121 | cropped from 800 — slower *and* narrower |
| 1024×768 | 120 | native |
| 640×480 | 210 | native |
| **640×400** | **271** | true 0.5× rescale of 1280×800 — the only mode whose intrinsics scale |
| 320×240 | 422 | native |

In [1]:
import sys, time, threading
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as W
from IPython.display import display

import elp
import sources

import subprocess

In [2]:
HERE = Path.cwd()
if HERE.name != "camera":                    # tolerate running from the repo root
    HERE = next(p for p in [HERE / "controller/camera",
                            HERE / "ESP32_PMW/controller/camera"] if p.exists())
sys.path[:0] = [str(HERE)]

# Captures land beside the ones already in the repo, one directory per session so a
# run is never mixed with an earlier one.
CAPTURES = HERE.parent / "pose" / "assets" / "captures"

print("modes on file:", ", ".join(f"{m.width}x{m.height}" for m in elp.modes()))
print("captures ->", CAPTURES)

modes on file: 1280x800, 1280x720, 1024x768, 800x600, 640x480, 640x400, 320x240, 160x120
captures -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/controller/pose/assets/captures


In [3]:
# print ffmpeg camera names
print(subprocess.run(["ffmpeg", "-hide_banner", "-f", "avfoundation",
                      "-list_devices", "true", "-i", ""],
                     capture_output=True, text=True).stderr)

# print camera indeices
for i, w, h in elp.probe_indices(max_index=4):
    print(f"  opencv index {i}: {w}x{h}")

[AVFoundation indev @ 0x9c540c140] AVFoundation video devices:
[AVFoundation indev @ 0x9c540c140] [0] FaceTime HD Camera
[AVFoundation indev @ 0x9c540c140] [1] Global Shutter Camera
[AVFoundation indev @ 0x9c540c140] [2] Global Shutter Camera
[AVFoundation indev @ 0x9c540c140] [3] Kevin’s iPhone Camera
[AVFoundation indev @ 0x9c540c140] [4] Kevin’s iPhone Desk View Camera
[AVFoundation indev @ 0x9c540c140] [5] Capture screen 0
[AVFoundation indev @ 0x9c540c140] AVFoundation audio devices:
[AVFoundation indev @ 0x9c540c140] [0] MacBook Pro Microphone
[AVFoundation indev @ 0x9c540c140] [1] Kevin’s iPhone Microphone
[AVFoundation indev @ 0x9c540c140] [2] Microsoft Teams Audio
[AVFoundation indev @ 0x9c540c140] [3] ZoomAudioDevice
[AVFoundation indev @ 0x9c540c140] [4] KTMA-2
[in#0 @ 0x9c540c000] Error opening input: Input/output error
Error opening input file .
Error opening input files: Input/output error

  opencv index 0: 1280x800
  opencv index 1: 1280x800
  opencv index 2: 1920x1080


## The preview widget

One helper, used by both blocks below. It takes any source that `elp.open_group` returns —
one camera or a stereo pair — and does not care which, because `elp.as_frames` normalises a
read to `(t, [frame, ...])`. **Capture** writes a still, **Record** writes a clip.

Things worth knowing about it:

- **The preview thread never writes files.** It keeps the newest frame in a slot; the button
  copies whatever is in that slot. So a capture is always a frame that was actually
  displayed, and pressing the button cannot stall the preview.
- **The JPEG encode is throttled** to `hz`. Encoding at 271 fps costs more than the capture
  does and would make the display rate the thing you are measuring.
- **Stereo frames are saved with the same index and their measured skew**, not assumed to be
  simultaneous. Two free-running USB cameras are not synchronised, and a pair whose skew you
  did not record is a pair you cannot use for calibration later.
- **A clip is buffered in RAM and encoded when you finish it**, never inside the read loop.
  FFV1 encodes at ~40 fps at 1280×800 and ~171 fps at 640×400 while the camera delivers 121
  and 271, so encoding inline would back-pressure the grabber and lose most of the take.
  Budget ~1.0 MB per frame at 1280×800, ~0.26 MB at 640×400, per camera; `max_rec_mb`
  (default 2000) closes the clip rather than filling memory, and **Stop** finishes a clip
  in progress instead of discarding it.
- **FFV1 in Matroska, mono in as mono** — verified bit-exact on this machine, 30 frames in
  and 30 identical frames back. A lossy codec would put ringing on the black-to-white marker
  edge, which is the same reason the stills are PNG.

### Playback rate and slow motion

`play_fps` sets the container rate and nothing else — it never resamples, so every frame
recorded is a frame written, and no frame is duplicated or dropped to hit a target.

- `play_fps=None` — the clip is written at the *measured* rate, so it plays in real time.
- `play_fps=30` — a 271 fps take plays back **9× slow**; a 121 fps take, 4× slow. The status
  line prints the factor when the clip is written.

Either way `<clip>_times.csv` holds the true per-frame stamps, so the real rate survives even
when the container is deliberately lying about it, and the interval is never exactly uniform.

In [4]:
class Preview:
    """
    Live preview with capture and record buttons. Works for one camera or a pair.
    """

    def __init__(self, source, cams, outdir, hz=15.0, quality=85, scale=0.5,
                 play_fps=None, max_rec_mb=2000):
        self.source, self.cams, self.hz, self.quality, self.scale = source, cams, hz, quality, scale
        self.play_fps = play_fps     # None -> real time; a number -> slow motion
        self.max_rec_mb = max_rec_mb
        self.outdir = Path(outdir)
        self.outdir.mkdir(parents=True, exist_ok=True)
        self._latest = None          # (t, [frame, ...]) -- newest read, for the button
        self._rec = None             # every read while a clip runs, else None
        self._rec_bytes = 0
        self._stop = threading.Event()
        self._thread = None
        self.n_saved = 0
        self.n_clips = 0

        self.image = W.Image(format="jpeg")
        self.status = W.HTML("<i>starting…</i>")
        self.btn_shot = W.Button(description="Capture", button_style="primary", icon="camera")
        self.btn_rec = W.Button(description="Record", button_style="warning", icon="video-camera")
        self.btn_stop = W.Button(description="Stop", button_style="danger", icon="stop")
        self.btn_shot.on_click(lambda _b: self.capture())
        self.btn_rec.on_click(lambda _b: self.record())
        self.btn_stop.on_click(lambda _b: self.stop())
        self.ui = W.VBox([self.image,
                          W.HBox([self.btn_shot, self.btn_rec, self.btn_stop]), self.status])

    # -- display -------------------------------------------------------------
    def _tile(self, frames):
        """
        One frame, or two side by side with a divider so they cannot be confused.
        """

        shown = [f if f.ndim == 2 else cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
        if self.scale != 1.0:
            shown = [cv2.resize(f, None, fx=self.scale, fy=self.scale,
                                interpolation=cv2.INTER_AREA) for f in shown]
        if len(shown) == 1:
            return shown[0]
        gap = np.full((shown[0].shape[0], 4), 255, np.uint8)
        return np.hstack([shown[0], gap, shown[1]])

    def _loop(self):
        period = 1.0 / self.hz
        t_drawn = 0.0
        while not self._stop.is_set():
            item = elp.as_frames(self.source.read())
            if item is None:
                time.sleep(0.005)
                continue
            self._latest = item
            if self._rec is not None:
                # Buffer only. Appending costs nothing; encoding here would not.
                self._rec.append(item)
                self._rec_bytes += sum(f.nbytes for f in item[1])
                if self._rec_bytes > self.max_rec_mb * 1e6:
                    self.record()        # close the clip rather than fill memory
            now = time.monotonic()
            if now - t_drawn < period:
                continue
            t_drawn = now
            ok, buf = cv2.imencode(".jpg", self._tile(item[1]),
                                   [cv2.IMWRITE_JPEG_QUALITY, self.quality])
            if ok:
                self.image.value = buf.tobytes()
            self._render_status()

    def _render_status(self):
        bits = [f"{c.actual['width']}x{c.actual['height']}" for c in self.cams]
        drops = sum(getattr(c, "n_dropped", 0) for c in self.cams)
        skew = ""
        if len(self.cams) > 1 and hasattr(self.source, "skew_stats"):
            # Already in ms, and cumulative over the session -- see sources.skew_stats.
            st = self.source.skew_stats()
            if st:
                skew = (f" &middot; skew {st['median_ms']:.1f} ms median,"
                        f" {st['max_ms']:.1f} worst")
        rec = ""
        if self._rec is not None:
            rec = (f" &middot; <b style='color:#c00'>REC</b> {len(self._rec)} frames,"
                   f" {self._rec_bytes / 1e6:.0f}/{self.max_rec_mb} MB")
        self.status.value = (f"{' + '.join(bits)} &middot; saved <b>{self.n_saved}</b>"
                             f" &middot; clips <b>{self.n_clips}</b>"
                             f" &middot; dropped {drops}{skew}{rec}")

    # -- actions -------------------------------------------------------------
    def start(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        display(self.ui)
        return self

    def capture(self):
        item = self._latest
        if item is None:
            self.status.value = "<b>nothing to capture</b> — no frame has arrived yet"
            return
        t, frames = item
        idx = self.n_saved
        names = []
        for c, frame in enumerate(frames):
            p = self.outdir / (f"{idx:03d}.png" if len(frames) == 1 else f"{idx:03d}_cam{c}.png")
            cv2.imwrite(str(p), frame)
            names.append(p.name)
        self.n_saved += 1
        self._render_status()

    def record(self):
        """
        Start a clip, or finish the one running and write it out.

        Frames are buffered and encoded afterwards, never inside the read loop:
        FFV1 encodes at ~40 fps at 1280x800 while the camera delivers 121, so
        encoding inline would back-pressure the grabber and lose most of the take.
        A pair stays a pair -- both files take frame *i* from one stereo read.

        `play_fps` is the container rate only, and never resamples: every frame
        recorded is a frame written. Leave it `None` and the clip plays at the
        rate it was shot; set it to 30 and a 271 fps take plays 9x slow. The true
        stamps go to `<clip>_times.csv` either way, so the slow-motion factor
        never has to be guessed back out of the file.
        """

        if self._rec is None:
            self._rec, self._rec_bytes = [], 0
            self.btn_rec.description, self.btn_rec.button_style = "Finish clip", "danger"
            self._render_status()
            return

        buf, self._rec = self._rec, None      # stop buffering before the slow part
        self.btn_rec.description, self.btn_rec.button_style = "Record", "warning"
        if len(buf) < 2:
            self.status.value = "<b>clip discarded</b> — fewer than two frames"
            return

        t = np.array([s for s, _ in buf]) - buf[0][0]
        fps_real = (len(buf) - 1) / t[-1]
        fps_out = self.play_fps or fps_real
        stem, n_cam = f"clip{self.n_clips:02d}", len(buf[0][1])
        for c in range(n_cam):
            h, w = buf[0][1][c].shape[:2]
            name = f"{stem}.mkv" if n_cam == 1 else f"{stem}_cam{c}.mkv"
            vw = cv2.VideoWriter(str(self.outdir / name),
                                 cv2.VideoWriter_fourcc(*"FFV1"), fps_out, (w, h), False)
            for _, frames in buf:
                vw.write(frames[c])
            vw.release()
        np.savetxt(self.outdir / f"{stem}_times.csv", t, fmt="%.6f",
                   header="t_s, seconds since the first frame of the clip", comments="# ")
        self.n_clips += 1
        slowed = (f" &middot; written at {fps_out:g} fps "
                  f"(<b>{fps_real / fps_out:.1f}x slow</b>)") if self.play_fps else ""
        self.status.value = (f"wrote <b>{stem}</b> &middot; {len(buf)} frames x {n_cam} cam "
                             f"&middot; shot at {fps_real:.1f} fps{slowed}")

    def stop(self):
        if self._rec is not None:
            self.record()                # never lose a clip to pressing Stop
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=2.0)
        try:
            self.source.close()
        finally:
            self.btn_shot.disabled = self.btn_rec.disabled = self.btn_stop.disabled = True
            self.status.value = (f"stopped &middot; <b>{self.n_saved}</b> capture(s), "
                                 f"<b>{self.n_clips}</b> clip(s) in <code>{self.outdir}</code>")

## 1. Single Camera

In [9]:
INDEX    = 0             # set by hand from the list above
MODE     = "640x400"
PLAY_FPS = 271          # None -> clips play at the rate they were shot; 30 -> slow motion

session = CAPTURES / f"elp_{time.strftime('%Y%m%d_%H%M%S')}"
src, cams = elp.open_group(INDEX, mode=MODE)
print(f"camera {INDEX}: asked {MODE}, got {cams[0].actual['width']}x{cams[0].actual['height']}")
print("session ->", session)

mono = Preview(src, cams, session, play_fps=PLAY_FPS).start()

camera 0: asked 640x400, got 640x400
session -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/controller/pose/assets/captures/elp_20260817_134741


## 2. Stereo Pair

In [8]:
INDICES     = [0, 1]     # set by hand from the list above
MODE_STEREO = "1280x800"
PLAY_FPS_S  = None       # None -> real time; 30 -> slow motion, both files alike
MAX_SKEW_S  = None       # None keeps every pair; set e.g. 0.010 to reject the worst

session_s = CAPTURES / f"stereo_{time.strftime('%Y%m%d_%H%M%S')}"
pair, pair_cams = elp.open_group(INDICES, mode=MODE_STEREO, max_skew_s=MAX_SKEW_S)
for i, c in zip(INDICES, pair_cams):
    print(f"camera {i}: {c.actual['width']}x{c.actual['height']}")
print("session ->", session_s)

stereo = Preview(pair, pair_cams, session_s, play_fps=PLAY_FPS_S).start()

camera 0: 1280x800
camera 1: 1280x800
session -> /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/controller/pose/assets/captures/stereo_20260823_131236


## 3. Summary

In [10]:
for name, pv in (("mono", globals().get("mono")), ("stereo", globals().get("stereo"))):
    if pv is None:
        continue
    files = sorted(pv.outdir.glob("*.png"))
    clips = sorted(pv.outdir.glob("*.mkv"))
    print(f"{name}: {len(files)} still(s), {len(clips)} clip file(s) in {pv.outdir}")
    st = pv.source.skew_stats() if hasattr(pv.source, "skew_stats") else {}
    if st:
        print(f"   skew over {st['n']} pairs: median {st['median_ms']:.2f} ms, "
              f"p95 {st['p95_ms']:.2f}, worst {st['max_ms']:.2f}   "
              f"(one frame at 121 fps = 8.3 ms; {st['dropped']} pairs rejected)")
    if files:
        f = cv2.imread(str(files[-1]), cv2.IMREAD_UNCHANGED)
        print(f"   last still: {files[-1].name}  {f.shape}  {f.dtype}  "
              f"levels p5 {np.percentile(f, 5):.0f} / median {np.median(f):.0f} / p95 {np.percentile(f, 95):.0f}")
    for c in clips:
        # Read each clip back rather than trusting what was meant to be written.
        cap = cv2.VideoCapture(str(c))
        n, container = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        t = np.loadtxt(pv.outdir / f"{c.stem.split('_cam')[0]}_times.csv")
        real = (len(t) - 1) / t[-1] if len(t) > 1 else 0.0
        slowed = f", {real / container:.1f}x slow" if container else ""
        print(f"   {c.name}: {n} frames, {c.stat().st_size / 1e6:.1f} MB, "
              f"shot {real:.1f} fps, container {container:.1f} fps{slowed}")

mono: 0 still(s), 1 clip file(s) in /Users/meli/Desktop/Kevin/UCB/FlyingRobotsLinLiweiLab/ESP32_PMW/controller/pose/assets/captures/elp_20260817_134741
   clip00.mkv: 3037 frames, 224.1 MB, shot 207.9 fps, container 271.0 fps, 0.8x slow
